In [3]:
import pandas as pd
import json
import lseg.data as ld
from lseg.data.content import search
ld.open_session()

<lseg.data.session.Definition object at 0x1da1062a210 {name='workspace'}>

In [71]:
cpacs = pd.read_csv("../data/cpac/corporate_pacs_2004_2024.csv", dtype={"PermID":str})
RICs = cpacs["RIC"].dropna().unique().tolist()
PermIDs = cpacs["PermID"].dropna().unique().tolist()

In [131]:
with open('data_fields.json', 'r') as f:
    data_fields = json.load(f)
    fields = list(data_fields.keys())
    col_names = list(data_fields.values())

In [73]:
def collapse(x):
    x = [v for v in x if pd.notnull(v) and v != '']
    return x if x else None

In [141]:
df_RICs = ld.get_data(
    universe = RICs,
    fields = fields
)

c:\Users\zih028\AppData\Local\miniconda3\envs\python312\Lib\site-packages\lseg\data\_tools\_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


In [142]:
df_PermIDs = ld.get_data(
    universe = PermIDs,
    fields = fields
)

c:\Users\zih028\AppData\Local\miniconda3\envs\python312\Lib\site-packages\lseg\data\_tools\_dataframe.py:192:FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


In [143]:
df_RICs.columns = ["Instrument"] + col_names
df_PermIDs.columns = ["Instrument"] + col_names

In [144]:
collapse_cols = ["TRBC_ID", "TRBC_Econ_Sector", "TRBC_Business_Sector", "TRBC_Industry_Group",
                 "TRBC_Industry", "TRBC_Activity", "NAICS_ID", "NAICS_Sector", "NAICS_Subsector",
                 "NAICS_Industry_Group", "NAICS_International_Industry", "NAICS_National_Industry"]
agg_dict = {
    col: collapse if col in collapse_cols else 'first'
    for col in df_RICs.columns if col != 'Instrument'
}

In [145]:
df_RICs = df_RICs.groupby("Instrument", as_index=False).agg(agg_dict)
df_PermIDs = df_PermIDs.groupby("Instrument", as_index=False).agg(agg_dict)

In [146]:
df_all = pd.concat([df_RICs, df_PermIDs])

In [147]:
df_RICs.to_csv("../data/firms/public_firms.csv", index=False)
df_PermIDs.to_csv("../data/firms/private_firms.csv", index=False)
df_all.to_csv("../data/firms/all_firms.csv", index=False)

In [151]:
df_RICs["NAICS_Industry_Group"].apply(type).unique()

array([<class 'list'>, <class 'NoneType'>], dtype=object)